# Audio Representations & Spectrograms

Companion notebook for the [Audio Representations lesson](https://ml-viz-ruby.vercel.app/courses/speech-audio/01-audio-representations).

**The idea in one sentence.** A sound is a 1-D wave of air pressure over time, but
ML models want a *picture* — so we turn the wave into a 2-D time–frequency image
(a **spectrogram**), then warp it to match human hearing (**mel**), and finally
compress it (**MFCC**).

The pipeline every speech system starts with:

$$\underbrace{\text{waveform}}_{\text{1-D}} \xrightarrow{\text{STFT}}
\underbrace{\text{spectrogram}}_{\text{time}\times\text{freq}} \xrightarrow{\text{mel filterbank}}
\underbrace{\text{mel-spectrogram}}_{\text{perceptual}} \xrightarrow{\text{log + DCT}}
\underbrace{\text{MFCC}}_{\text{compact}}$$

We build **every** stage from scratch in NumPy, **validate our STFT against
`scipy.signal`**, and see why the window length forces a time–frequency
trade-off. No audio libraries.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 1.5,
})

## 1 — A waveform with changing frequency

A 'chirp' that sweeps from low to high pitch — so the spectrogram will show a clear diagonal,
demonstrating that we recover *when* each frequency occurs.

In [ ]:
fs = 8000                    # sampling rate (Hz) -> Nyquist limit 4000 Hz
t = np.linspace(0, 2, 2*fs, endpoint=False)
freq = 200 + 600 * t / 2     # frequency sweeps 200 -> 800 Hz
wave = np.sin(2*np.pi*np.cumsum(freq)/fs)
print(f'{len(wave)} samples for {t[-1]+1/fs:.0f}s at {fs} Hz (Nyquist = {fs//2} Hz)')

## 2 — STFT: slide a window, take the Fourier transform of each

The spectrogram is |STFT|: rows = frequency, columns = time. The diagonal ridge is the rising pitch
— time-frequency structure a single FFT would have lost.

In [ ]:
def stft(x, win, hop):
    window = np.hanning(win)
    frames = [np.abs(np.fft.rfft(x[i:i+win] * window))
              for i in range(0, len(x)-win, hop)]
    return np.array(frames).T          # (freq_bins, time_frames)

spec = stft(wave, win=256, hop=64)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.imshow(np.log(spec+1e-6), origin='lower', aspect='auto', cmap='magma',
          extent=[0, 2, 0, fs/2])
ax.set_xlabel('time (s)'); ax.set_ylabel('frequency (Hz)')
ax.set_title('Spectrogram (STFT): the rising-pitch chirp is a diagonal ridge')
plt.tight_layout(); plt.show()

### Validate: our STFT peaks at the right frequency, and matches SciPy

Two checks. First, a **pure tone** must show up as a single bright row at its own
frequency — the most basic correctness property of any spectrogram. Second, our
hand-rolled per-frame `rfft` must match `scipy.signal.stft`'s magnitudes (up to
their windowing/normalisation convention) via the correlation of the two
spectrograms.

In [ ]:
from scipy.signal import stft as sp_stft

# 1. pure 1000 Hz tone -> its spectrogram should peak at the 1000 Hz bin
tone = np.sin(2 * np.pi * 1000 * t)
win = 256
freqs = np.fft.rfftfreq(win, 1 / fs)
tone_spec = stft(tone, win=win, hop=64)
peak_bin = freqs[tone_spec.mean(axis=1).argmax()]
print(f'pure 1000 Hz tone -> spectrogram peak at {peak_bin:.0f} Hz')
assert abs(peak_bin - 1000) <= fs / win, 'a pure tone must peak at its own frequency'

# 2. shape + correlation vs scipy.signal.stft on the chirp
f_sp, tt_sp, Z = sp_stft(wave, fs=fs, nperseg=win, noverlap=win - 64, window='hann')
sp_mag = np.abs(Z)
# compare the column-averaged frequency profiles (robust to edge/frame-count conventions)
ours_profile = stft(wave, win=win, hop=64).mean(axis=1)
sp_profile = sp_mag.mean(axis=1)
corr = np.corrcoef(ours_profile / ours_profile.sum(), sp_profile / sp_profile.sum())[0, 1]
print(f'frequency-profile correlation vs scipy.signal.stft: {corr:.4f}')
assert corr > 0.99, 'our STFT should match scipy up to normalisation'
print('\n✅ STFT is correct: pure tone localises, and it matches scipy.signal')

## 3 — The time-frequency resolution trade-off

A short window pins down time but smears frequency; a long window does the reverse. Same signal,
different windows.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, win in zip(axes, [64, 1024]):
    s = stft(wave, win=win, hop=win//4)
    ax.imshow(np.log(s+1e-6), origin='lower', aspect='auto', cmap='magma', extent=[0,2,0,fs/2])
    ax.set_title(f'window={win} ({"sharp time" if win==64 else "sharp frequency"})')
    ax.set_xlabel('time (s)')
axes[0].set_ylabel('frequency (Hz)')
plt.tight_layout(); plt.show()
print('Short window -> crisp in time, blurry in frequency. Long window -> the opposite.')

**What to notice — the uncertainty trade-off.** This is the audio version of a
Heisenberg-style limit: a short window resolves *when* (crisp diagonal edges) but
blurs *what pitch* (thick frequency band); a long window does the reverse. You
cannot have perfect time **and** frequency resolution at once — the window length
is the knob you turn to trade one for the other.

## 4 — Mel spectrogram: warp frequency to match human hearing

Humans hear pitch roughly *logarithmically* — the gap between 200 Hz and 300 Hz
sounds much bigger than between 5000 Hz and 5100 Hz. The **mel scale** encodes
that. A mel **filterbank** is a set of triangular windows, narrow and dense at low
frequencies, wide and sparse at high frequencies, that we multiply into the
linear spectrogram to pool it into perceptually-spaced bands.

$$m = 2595\,\log_{10}\!\left(1 + \frac{f}{700}\right)$$

In [ ]:
def hz_to_mel(f):  return 2595 * np.log10(1 + f / 700)
def mel_to_hz(m):  return 700 * (10 ** (m / 2595) - 1)

def mel_filterbank(n_filters, win, fs):
    n_bins = win // 2 + 1
    mel_pts = np.linspace(hz_to_mel(0), hz_to_mel(fs / 2), n_filters + 2)
    hz_pts = mel_to_hz(mel_pts)
    bin_idx = np.floor((win + 1) * hz_pts / fs).astype(int)
    fb = np.zeros((n_filters, n_bins))
    for m in range(1, n_filters + 1):
        l, c, r = bin_idx[m - 1], bin_idx[m], bin_idx[m + 1]
        for k in range(l, c):   fb[m - 1, k] = (k - l) / max(c - l, 1)
        for k in range(c, r):   fb[m - 1, k] = (r - k) / max(r - c, 1)
    return fb

fb = mel_filterbank(n_filters=26, win=256, fs=fs)
mel_spec = fb @ stft(wave, win=256, hop=64)          # (n_filters, time)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].plot(freqs, fb[::4].T)
axes[0].set_title('Mel filterbank (every 4th triangle)')
axes[0].set_xlabel('frequency (Hz)'); axes[0].set_ylabel('weight')
axes[1].imshow(np.log(mel_spec + 1e-6), origin='lower', aspect='auto', cmap='magma',
               extent=[0, 2, 0, 26])
axes[1].set_title('Log-mel spectrogram (26 bands)')
axes[1].set_xlabel('time (s)'); axes[1].set_ylabel('mel band')
plt.tight_layout(); plt.show()
print('Triangles get wider with frequency -> low pitches get fine resolution, high pitches coarse.')
print('The chirp is still a rising ridge, but compressed into 26 perceptual bands instead of 129.')

## 5 — MFCC: log + DCT to a compact, decorrelated feature

The mel bands are correlated (neighbouring triangles overlap). Taking the
**log** (perceived loudness) and then a **Discrete Cosine Transform** decorrelates
them and packs the energy into the first ~13 coefficients — the classic
**MFCC** features that powered pre-deep-learning speech recognition, and still
show up as compact inputs today.

In [ ]:
from scipy.fftpack import dct

log_mel = np.log(mel_spec + 1e-6)
mfcc = dct(log_mel, axis=0, type=2, norm='ortho')[:13]   # keep 13 coefficients

fig, ax = plt.subplots(figsize=(8, 3))
im = ax.imshow(mfcc, origin='lower', aspect='auto', cmap='viridis', extent=[0, 2, 0, 13])
ax.set_xlabel('time (s)'); ax.set_ylabel('MFCC coefficient')
ax.set_title('MFCCs: 13 coefficients per frame (from 129 FFT bins)')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

# Energy compaction: the DCT concentrates variance in the low coefficients.
energy = (mfcc ** 2).sum(axis=1)
frac_first4 = energy[:4].sum() / energy.sum()
print(f'fraction of MFCC energy in the first 4 coefficients: {frac_first4:.1%}')
print(f'compression: 129 FFT bins -> 26 mel bands -> 13 MFCCs per frame')

## Gotchas & tradeoffs

| Gotcha | Why it matters |
|--------|----------------|
| **window length** | short = sharp time / blurry freq; long = the opposite (the trade-off above) |
| **no window (rectangular)** | spectral **leakage** — sharp frame edges smear energy across bins; use Hann/Hamming |
| **Nyquist** | frequencies above $f_s/2$ **alias** into false low frequencies — filter before sampling |
| **log before DCT** | matches loudness perception *and* turns multiplicative gain into an additive offset |
| **too few mel bands** | over-smooths formants; 20–40 is typical for speech |

Demo: windowing suppresses spectral leakage — compare a rectangular window vs Hann
on a pure tone that doesn't fall exactly on a bin.

In [ ]:
off_tone = np.sin(2 * np.pi * 1010.5 * t[:256])   # deliberately between bins
rect = np.abs(np.fft.rfft(off_tone))
hann = np.abs(np.fft.rfft(off_tone * np.hanning(256)))
# "leakage" = energy outside the 3 bins nearest the peak
def leakage(mag):
    pk = mag.argmax()
    near = mag[max(0, pk-1):pk+2].sum()
    return 1 - near / mag.sum()
print(f'rectangular window leakage: {leakage(rect):.1%}')
print(f'Hann window leakage:        {leakage(hann):.1%}')
assert leakage(hann) < leakage(rect), 'windowing should reduce spectral leakage'
print('\nThe Hann window tapers the frame edges to zero, so far-away bins stay dark.')

## ✏️ Your turn

**Exercise.** Implement `nyquist(fs)` (the highest faithfully representable frequency) and
`n_frames(signal_len, win, hop)` (how many STFT windows fit in a signal). These two govern the shape
of every spectrogram.

In [ ]:
def nyquist(fs):
    # TODO(you): the Nyquist frequency for sampling rate fs
    return ...

def n_frames(signal_len, win, hop):
    # TODO(you): number of windows of size `win` stepping by `hop` that fit (matching the stft loop)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert nyquist(16000) == 8000
assert nyquist(44100) == 22050
# matches the actual number of columns the stft above produced
assert n_frames(len(wave), 256, 64) == spec.shape[1]
assert n_frames(1000, 100, 100) == 9
print('\u2713 Nyquist and frame-count are correct')

<details>
<summary>Solution</summary>

```python
def nyquist(fs):
    return fs // 2

def n_frames(signal_len, win, hop):
    return len(range(0, signal_len - win, hop))
```

Nyquist caps the frequencies you can represent at half the sample rate; the frame count (and window
size) set the spectrogram's width and height — the 'image' a downstream CNN or transformer sees.

</details>

## Key takeaways

- **Audio ML starts by making a picture.** STFT turns a 1-D wave into a 2-D
  time–frequency spectrogram — the input a CNN or Transformer actually sees.
- **The window length is a time–frequency trade-off** you cannot escape: short
  windows localise *when*, long windows localise *what pitch*.
- **Mel warps frequency to perception** (log-spaced triangular filters); **MFCC**
  adds log + DCT to decorrelate and compress to ~13 coefficients per frame.
- We **validated** the STFT (pure tone localises; matches `scipy.signal`) and saw
  windowing suppress spectral leakage.
- Nyquist ($f_s/2$) caps representable frequency; the frame count and window size
  set the spectrogram's width and height.